# Notebook 09 — Keystone Analysis: Addressee vs Co-occurrence (Pillar B)

**Requires:** run `09_run_orchestrator` first — this notebook reads from `tables_n17/`

**Role:** Section 6 of thesis — *Structural Payoff: when addressee tagging changes the answer.*

Reads the keystone comparison table from the orchestrator, adds:
- Fisher exact test on keystone asymmetry
- Note on Inside Out co-occurrence artefact
- female_alter_betw_z descriptive comparison

**N=17:** Soul dropped, Sulley kept.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from scipy import stats

CLEAN  = Path('..').resolve()
sys.path.insert(0, str(CLEAN / 'analysis' / 'h1_homophily'))
TBL    = CLEAN / 'analysis' / 'h1_homophily' / 'tables_n17'
DATA   = CLEAN / 'data' / 'processed'

from _common import GENDER_PALETTE, set_style, cliffs_delta, mannwhitney, perm_test_diff, bootstrap_diff_ci, power_note
set_style()

DROP_FILMS  = {'soul_2020'}
DROP_PROTAG = ('monsters_inc_2001', 'Mike')

def apply_conventions(df):
    df = df[~df['film_id'].isin(DROP_FILMS)].copy()
    df = df[~((df['film_id']==DROP_PROTAG[0]) & (df['protagonist']==DROP_PROTAG[1]))].copy()
    return df.reset_index(drop=True)

print('Setup OK')

## 1. Keystone agreement table (from orchestrator `step2f_keystone_agreement.csv`)

In [ ]:
ks = pd.read_csv(TBL / 'step2f_keystone_agreement.csv')
print(f'Loaded {len(ks)} films')
print(ks[['film_title','lead_gender','keystone_addr','keystone_addr_gender',
          'keystone_cooc','keystone_cooc_gender','keystone_agree','gender_flip']].to_string(index=False))
ks.to_csv(TBL / 'nb09_keystone_agreement.csv', index=False)

## 2. Agreement stats and gender-flip pattern

In [ ]:
n = len(ks)
print(f'Overall agreement: {ks.keystone_agree.sum()}/{n} = {ks.keystone_agree.mean():.1%}')
print()

for method, col in [('Addressee','keystone_addr_gender'),('Co-occurrence','keystone_cooc_gender')]:
    ks['cross'] = ks.apply(lambda r: r[col]!=r['lead_gender'] if pd.notna(r[col]) else False, axis=1)
    ct = ks.groupby('lead_gender')['cross'].agg(['sum','count'])
    ct.columns=['cross_gender','n']; ct['pct']=(ct.cross_gender/ct.n*100).round(1)
    print(f'{method} — cross-gender keystone:'); print(ct); print()

flips = ks[ks['gender_flip']==1]
print(f'Gender flips: {len(flips)}/{n}')
print(flips[['film_title','lead_gender','keystone_addr','keystone_addr_gender',
             'keystone_cooc','keystone_cooc_gender']].to_string(index=False))

## 3. Fisher exact test — keystone asymmetry

In [ ]:
ks['addr_cross'] = ks.apply(
    lambda r: r['keystone_addr_gender']!=r['lead_gender'] if pd.notna(r['keystone_addr_gender']) else False,
    axis=1)

ct = pd.crosstab(ks['lead_gender'], ks['addr_cross'])
print('Contingency table (lead_gender × cross-gender keystone, addressee):')
print(ct); print()
odds, p = stats.fisher_exact(ct.values)
print(f'Fisher exact (two-sided): odds={odds:.3f}  p={p:.4f}')
print('→ Significant at α=0.05' if p<0.05 else '→ Not significant')

## 4. Note on Inside Out co-occurrence keystone

The orchestrator reports **3 gender flips** including Inside Out (addr=Anger M, cooc=Disgust F).

However: in the Inside Out co-occurrence network, **all non-protagonist characters have
betweenness = 0.000** — the emotions share nearly every scene → near-complete graph → no
structural bridges. When all values tie at zero, `max()` returns the first node by
dict-iteration order — an implementation artefact, not a structural result.

**Robust flips: 2 (Zootopia, Encanto).** Inside Out should be treated as indeterminate.
All robust flips are F-led, all in the same direction (addr=M, cooc=F).

## 5. female_alter_betw_z — from orchestrator `step4_female_alter_betw_z.csv`

In [ ]:
fa = pd.read_csv(TBL / 'step4_female_alter_betw_z.csv')
fa = apply_conventions(fa)

F_v = fa[fa['lead_gender']=='F']['female_alter_betw_z'].dropna().values
M_v = fa[fa['lead_gender']=='M']['female_alter_betw_z'].dropna().values

print(f'n_F={len(F_v)}  n_M={len(M_v)}')
print(f'Median F={np.median(F_v):+.3f}  Median M={np.median(M_v):+.3f}')
print(f"Cliff's delta = {cliffs_delta(F_v, M_v):+.3f}")
mw = mannwhitney(F_v, M_v)
print(f"Mann-Whitney p = {mw['p']:.4f}")
pm = perm_test_diff(F_v, M_v, stat=np.median)
print(f"Permutation p = {pm['p_two_sided']:.4f}")
print(power_note(len(F_v), len(M_v)))

## 6. Conventions & reproducibility

| | Choice |
|---|---|
| Soul (2020) | Dropped |
| Monsters Inc | Sulley only |
| N | 17 (9F + 8M) |
| Source tables | `tables_n17/` generated by `n17_orchestrator.py` via `09_run_orchestrator` |
| Fisher test | Exact (not asymptotic) |

See **`09b_keystone_dirw_experiment`** for the directed+weighted keystone variant.
See **`10_descriptive_exploratory`** for all descriptive metrics with verdicts.